# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kuteesatendojeremiah/Tendojerry-Flyrank/blob/main/work/notebooks/w07_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip install -q duckdb huggingface_hub

import os, getpass
import duckdb
import numpy as np
import pandas as pd

HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients":       f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content":       f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily":        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d":    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

# Same window as ML-04/05/06/07 (w03-w06) — mid-panel month, never the sealed June 2026 sample.
MONTH_START = "2026-03-01"
MONTH_END_EXCL = "2026-04-01"      # half-open: report_date < MONTH_END_EXCL
PREV30_START = "2026-01-30"        # the 30 days immediately before MONTH_START
PREV30_END_EXCL = MONTH_START

print(f"Connected. Iterating on month={MONTH_START[:7]} | prev30 window: [{PREV30_START}, {PREV30_END_EXCL})")


Paste your Hugging Face READ token (hf_...): ··········
Connected. Iterating on month=2026-03 | prev30 window: [2026-01-30, 2026-03-01)


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

My lane's question is "which first?" — a ranking problem, so the method just needs to output a
score to sort by, evaluated at Precision@50 (`skills/training-honest-models/SKILL.md`'s method
table: ranking → any classifier's probability).

**Two methods, both trained fresh in this notebook (not cited from ML-05's earlier run) so the
comparison table in Section 3 is computed in one place, on one split:**

1. **Logistic Regression** — the readable starting point. ML-05 (`w04`) already ran this exact
   feature vector through it and got a client-grouped Precision@50 of 0.440 (1.38x lift) — which
   *lost* to ML-07's rule baseline (0.540, 1.69x lift). Re-running it here isn't redundant: it
   puts the number in the same table, same split object, as everything else.
2. **Random Forest** — the "stronger" step the method table points to next. Since the linear
   model already fell short of the baseline, the real question this lane needs answered is
   whether a non-linear model can pick up interaction effects (e.g. CTR gap *combined with*
   content type or intent) that a linear model can't. A plain Decision Tree isn't run as a
   separate third method — Random Forest already subsumes it as an ensemble, and the
   interpretability a single tree would offer is covered instead by feature importances in
   Section 4.

**Features:** identical to ML-05 — 5 numeric prev30 features (`imp_prev30`, `clk_prev30`,
`avg_position_prev30`, `ctr_prev30`, `active_days_prev30`) + 4 categoricals (`content_type`,
`competition_level`, `main_intent`, `model_used`), same fills, same one-hot encoding. No new
features introduced here — this notebook is a method comparison against a fixed, already-vetted
feature vector, not a feature-engineering pass.

In [2]:
# --- Rebuild the identical feature vector as ML-05 (w04_feature_leakage_check.ipynb) ---
# Same prev30 numeric features, same NULLIF handling on avg_position, same fills.
feature_frame = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS imp_prev30,
           SUM(gsc_clicks) AS clk_prev30,
           AVG(NULLIF(gsc_avg_position, 0)) AS avg_position_prev30,
           COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS active_days_prev30
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '{PREV30_START}' AND report_date < DATE '{PREV30_END_EXCL}'
    GROUP BY 1, 2
""").df()
feature_frame["ctr_prev30"] = (feature_frame["clk_prev30"] / feature_frame["imp_prev30"]).fillna(0)

# Same categorical discovery as ML-05: safe, low-cardinality VARCHAR columns on dim_content.
content_schema = con.sql(f"DESCRIBE SELECT * FROM {TABLES['dim_content']}").df()
EXCLUDE_LIKE = ("client", "hash", "id", "profile", "account", "flag", "score")
candidate_cols = [
    c for c in content_schema.loc[content_schema["column_type"] == "VARCHAR", "column_name"]
    if not any(bad in c.lower() for bad in EXCLUDE_LIKE)
]
cat_features = []
for col in candidate_cols:
    n_distinct = con.sql(f"SELECT COUNT(DISTINCT {col}) FROM {TABLES['dim_content']}").fetchone()[0]
    if 1 < n_distinct <= 15:
        cat_features.append(col)
print(f"Categorical features (same 4 as ML-05 expected): {cat_features}")

cols_sql = ", ".join(cat_features)
content_meta = con.sql(f"SELECT content_hash_id, {cols_sql} FROM {TABLES['dim_content']}").df()
feature_frame = feature_frame.merge(content_meta, on="content_hash_id", how="left")
for col in cat_features:
    feature_frame[col] = feature_frame[col].fillna("unknown")
feature_frame = pd.get_dummies(feature_frame, columns=cat_features, prefix=cat_features)

# --- Label (identical formula to ML-04/05/06/07) ---
march = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp_march
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '{MONTH_START}' AND report_date < DATE '{MONTH_END_EXCL}'
    GROUP BY 1, 2
""").df()
data = feature_frame.merge(march, on=["client_hash_id", "content_hash_id"], how="inner")
data = data[data["imp_prev30"] > 0].copy()
data["is_declining"] = (data["imp_march"] < 0.8 * data["imp_prev30"]).astype(int)

feature_cols = [c for c in feature_frame.columns if c not in ("client_hash_id", "content_hash_id")]
print(f"\n{len(data):,} content items, {len(feature_cols)} feature columns, base rate {data['is_declining'].mean():.3f}")
print("Matches ML-05/ML-07's 146,253 rows / 0.285-0.320 base-rate range if the pipeline is consistent.")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Categorical features (same 4 as ML-05 expected): ['content_type', 'competition_level', 'main_intent', 'model_used']


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


146,253 content items, 22 feature columns, base rate 0.285
Matches ML-05/ML-07's 146,253 rows / 0.285-0.320 base-rate range if the pipeline is consistent.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Client-grouped**, not random: pages from the same client share a lot of structure (template,
niche, traffic tier), so a random row-level split lets a model partly memorize "this client's
pages behave like X" rather than learning something that generalizes to a client it has never
seen — exactly the risk `skills/hunting-leakage-and-validating/SKILL.md` and ML-05's addendum
flagged (random-split AUC 0.661 vs. client-grouped 0.629, a real but small +0.032 gap).

**Not time-aware** on top of that, because the label itself already encodes the only time
dimension this slice has (prev30 → March), and prev30/March don't overlap (checked in ML-05
Leakage test 2). There's no second, later window available in this development slice to hold
out chronologically without leaving the mid-panel month — that's what the sealed June sample is
for, at the very end of the capstone, not for routine model comparison.

**Same split object as ML-05 and ML-07**, not just the same method: identical
`GroupShuffleSplit(test_size=0.3, random_state=42)` grouped on `client_hash_id`. Because the
split depends only on the set of client IDs (not on row order or feature values), this
reproduces the exact same 29 train / 13 test client partition both of those notebooks used —
so Section 3's model numbers and ML-07's baseline number (0.540 Precision@50, 1.69x lift) are
comparable on the identical held-out clients, not just the same method.

In [3]:
from sklearn.model_selection import GroupShuffleSplit

X_full = data[feature_cols].fillna(0)
y_full = data["is_declining"]
groups = data["client_hash_id"]

# Identical params to ML-05's addendum and ML-07 Section 2 -> identical client partition,
# since the split depends only on the set of client_hash_id groups.
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X_full, y_full, groups=groups))

n_train_clients = groups.iloc[train_idx].nunique()
n_test_clients = groups.iloc[test_idx].nunique()
overlap = set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])
print(f"Client-grouped split: {n_train_clients} train clients, {n_test_clients} test clients, "
      f"overlap={len(overlap)} (must be 0 for an honest holdout)")
print("Expect 29 train / 13 test / 0 overlap to match ML-05's addendum and ML-07 Section 2 exactly.")

X_train, X_test = X_full.iloc[train_idx], X_full.iloc[test_idx]
y_train, y_test = y_full.iloc[train_idx], y_full.iloc[test_idx]
test_base_rate = y_test.mean()
print(f"\nTest-set base rate: {test_base_rate:.3f} (n_test={len(test_idx):,}) — "
      "compare to ML-07's held-out base rate of 0.320.")

Client-grouped split: 29 train clients, 13 test clients, overlap=0 (must be 0 for an honest holdout)
Expect 29 train / 13 test / 0 overlap to match ML-05's addendum and ML-07 Section 2 exactly.

Test-set base rate: 0.320 (n_test=75,733) — compare to ML-07's held-out base rate of 0.320.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

ML-07's rule (`ctr_gap x imp_prev30 x staleness_multiplier`) is recomputed here from scratch on
this notebook's own `data` and scored on the exact same held-out test rows the two models get
evaluated on — not cited from the earlier notebook's run — so all three numbers in the table
below come from one split, one metric (Precision@50), computed together in this cell.

In [4]:
# --- Recompute ML-07's baseline rule on this notebook's own data/split (not cited from w06) ---
# Position buckets + expected CTR per bucket, identical to w06_baseline_score.ipynb Section 1.
data["position_bucket"] = pd.cut(
    data["avg_position_prev30"], bins=[0, 3, 10, 20, 50, np.inf],
    labels=["top_3", "page_1", "striking", "page_3_5", "deep"]
)
ctr_by_position = data.dropna(subset=["position_bucket"]).groupby("position_bucket", observed=True).agg(
    total_clicks=("clk_prev30", "sum"), total_impressions=("imp_prev30", "sum")
)
ctr_by_position["expected_ctr"] = ctr_by_position["total_clicks"] / ctr_by_position["total_impressions"]
data["expected_ctr"] = data["position_bucket"].map(ctr_by_position["expected_ctr"]).astype(float)

# content_updated_date via .map (not merge) so row order/count of `data` can't shift and
# invalidate the positional train_idx/test_idx from Section 2. Same future-dating fix as ML-07:
# dim_content is current-state, not frozen at MONTH_START -- only trust dates on/before it.
content_dates = (con.sql(f"SELECT content_hash_id, content_updated_date FROM {TABLES['dim_content']}")
                  .df().drop_duplicates("content_hash_id").set_index("content_hash_id")["content_updated_date"])
data["content_updated_date"] = data["content_hash_id"].map(content_dates)
update_date = pd.to_datetime(data["content_updated_date"])
month_start_ts = pd.Timestamp(MONTH_START)
valid_update = update_date <= month_start_ts
data["days_since_update"] = np.where(valid_update, (month_start_ts - update_date).dt.days, np.nan)

VISIBILITY_FLOOR = 100
visible = data["imp_prev30"] >= VISIBILITY_FLOOR
has_position = data["position_bucket"].notna()
ctr_gap = (data["expected_ctr"] - data["ctr_prev30"]).clip(lower=0)
staleness_multiplier = 1 + data["days_since_update"].fillna(0) / 365
data["baseline_score"] = np.where(visible & has_position, ctr_gap * data["imp_prev30"] * staleness_multiplier, 0.0)

def precision_at_k(scores, labels, k=50):
    ranked = pd.DataFrame({"score": np.asarray(scores), "label": np.asarray(labels)}).sort_values("score", ascending=False)
    return ranked.head(k)["label"].mean()

# --- Train both models on the SAME train_idx/test_idx from Section 2 ---
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

lr = LogisticRegression(max_iter=2000, random_state=42).fit(X_train, y_train)
lr_scores_test = lr.predict_proba(X_test)[:, 1]

rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1).fit(X_train, y_train)
rf_scores_test = rf.predict_proba(X_test)[:, 1]

baseline_scores_test = data["baseline_score"].iloc[test_idx]

def summarize(name, scores, labels, base_rate):
    p50 = precision_at_k(scores, labels, k=50)
    return {
        "method": name, "n_test": len(labels), "base_rate": round(base_rate, 3),
        "precision_at_50": round(p50, 3),
        "lift": round(p50 / base_rate, 2) if base_rate > 0 else np.nan,
        "auc": round(roc_auc_score(labels, scores), 3),
    }

comparison = pd.DataFrame([
    summarize("ML-07 rule baseline", baseline_scores_test, y_test, test_base_rate),
    summarize("Logistic Regression", lr_scores_test, y_test, test_base_rate),
    summarize("Random Forest", rf_scores_test, y_test, test_base_rate),
]).set_index("method")

print(f"Same held-out test clients for all three rows (n_test={len(test_idx):,}, base_rate={test_base_rate:.3f}):\n")
print(comparison)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Same held-out test clients for all three rows (n_test=75,733, base_rate=0.320):

                     n_test  base_rate  precision_at_50  lift    auc
method                                                              
ML-07 rule baseline   75733       0.32             0.54  1.69  0.460
Logistic Regression   75733       0.32             0.42  1.31  0.629
Random Forest         75733       0.32             0.52  1.63  0.626


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

**To fill in after running this in Colab** (per `skills/training-honest-models/SKILL.md`'s "Read
the errors" checklist — a metric without error analysis is decoration):
- Which method actually won Section 3's table on this held-out set.
- The top 3 features it leans on (from the printed coefficients/importances below) and whether
  each plausibly relates to decline — or is suspiciously perfect (a leakage tell).
- Where the winning method is most wrong: which `content_type` / `position_bucket` groups
  collect the false positives in its top 50, and the 3 concrete wrong-case rows printed below.

In [5]:
# --- What does each fitted model lean on? ---
print("Logistic Regression coefficients (top 10 by |coefficient|):")
lr_coef = pd.Series(lr.coef_[0], index=X_train.columns).sort_values(key=abs, ascending=False)
print(lr_coef.head(10))

print("\nRandom Forest feature importances (top 10):")
rf_importance = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print(rf_importance.head(10))

print("\nML-07 rule baseline has no fitted weights to inspect -- it's transparent by design, "
      "formula visible in Section 3: ctr_gap x imp_prev30 x staleness_multiplier.")

# --- Error analysis on whichever method actually won Section 3's table ---
best_method = comparison["precision_at_50"].idxmax()
print(f"\nBest method by Precision@50 on this held-out set: {best_method}")

model_scores_map = {
    "ML-07 rule baseline": baseline_scores_test,
    "Logistic Regression": lr_scores_test,
    "Random Forest": rf_scores_test,
}
best_scores = model_scores_map[best_method]

test_df = data.iloc[test_idx].copy()
test_df["model_score"] = np.asarray(best_scores)
top50 = test_df.sort_values("model_score", ascending=False).head(50)
wrong_top50 = top50[top50["is_declining"] == 0]
print(f"{len(wrong_top50)} of the top 50 (by {best_method}) are false positives (flagged, not actually declining).")

if len(wrong_top50):
    type_cols = [c for c in test_df.columns if c.startswith("content_type_")]
    if type_cols:
        wrong_top50 = wrong_top50.copy()
        wrong_top50["content_type"] = wrong_top50[type_cols].idxmax(axis=1).str.replace("content_type_", "", regex=False)
        print("\nFalse positives by content_type:")
        print(wrong_top50["content_type"].value_counts())
    print("\nFalse positives by position_bucket:")
    print(wrong_top50["position_bucket"].value_counts(dropna=False))

    print("\n3 concrete wrong cases (in the model's top 50, but not actually declining):")
    show_cols = ["client_hash_id", "content_hash_id", "model_score", "imp_prev30", "ctr_prev30", "position_bucket", "is_declining"]
    print(wrong_top50[show_cols].head(3))
else:
    print("Every top-50 pick by the winning method was a true positive on this held-out set.")

Logistic Regression coefficients (top 10 by |coefficient|):
content_type_comparison article     -1.703855
content_type_feedly article          1.258564
model_used_gemini-2.5-flash         -0.476713
model_used_gemini-3-flash-preview   -0.403693
model_used_gpt-5-mini                0.292091
competition_level_HIGH              -0.245352
model_used_gpt-4o-mini               0.241377
main_intent_navigational            -0.235864
competition_level_MEDIUM            -0.235089
ctr_prev30                          -0.115938
dtype: float64

Random Forest feature importances (top 10):
avg_position_prev30                  0.389523
imp_prev30                           0.263295
active_days_prev30                   0.110972
ctr_prev30                           0.080283
clk_prev30                           0.033316
model_used_unknown                   0.015005
model_used_gemini-3-flash-preview    0.011755
content_type_feedly article          0.011131
model_used_gpt-4o-mini               0.010540
model_

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.